In [90]:
import json
from random import sample

import pandas as pd
from scipy import stats

# Check the exported JSON files

I've submitted the  [`submission.jso`](submission.json) and [`validation_submission.jso`](validation_submission.jso) files on 11 April 2025 with disappointing results.

| Description | Model Regression                      | Model Classification                   | Validation Score | Test Score | Expected |
|:------------|---------------------------------------|----------------------------------------|-----------------:|:-----------|---------:|
| small       | KAN([[7, 0], [1, 0], [2, 0]])         | KAN([[10, 0], [1, 1], [1, 0]])         |           0.3132 | 0.4836     |        - |
| big         | KAN([[7, 0], [2, 1], [1, 1], [2, 0]]) | KAN([[10, 0], [2, 1], [1, 1], [1, 0]]) |           0.3703 | 0.4644     |        - |
| fix + trick | KAN([[7, 0], [1, 0], [2, 0]])         | KAN([[10, 0], [1, 1], [1, 0]])         |                ? | ?          | 0.894779 |


In [83]:
# 4s
with open('train.json') as fp:
    train_y_hat: list[tuple[int, dict]] = list(json.load(fp).items())
train_y = pd.read_csv('../dataset/0-original/y.csv')

In [116]:
def score_rgr(pdf_type, pdf_args, true_target):
    # Extract the scipy model
    model = {'norm': stats.norm, 'cauchy': stats.cauchy}[pdf_type]

    # Calculate the score
    score = model.pdf(true_target, **pdf_args)

    # # The following block of code generates the full PDF for potential normalization
    #
    # # Generate x values for plotting and integration
    # n_points = 100000
    # x = np.linspace(-100, 100, n_points)
    # y = model.pdf(x, **pdf_args)
    #
    # # Verify the area under the curve using numerical integration
    # area = simps(y, x)
    #
    # # Normalize the PDF if the area greater than 1
    # if area > 1:
    #     score = score / area
    #
    # # Normalize score if the PDF max is greater than 1
    # y_max = y.max()

    # stats.norm optimization: the max y is always in the loc and the area is always 1
    y_max = model.pdf(pdf_args['loc'], **pdf_args)
    if y_max > 1:
        # y = y / y_max
        score = score / y_max

    return score

In [48]:
def score_cls(pred_label: int, confidence: int, true_label: int) -> int:
    # check confidence to make sure it's between 0 and 1
    if (confidence < 0) or (confidence > 1):
        return -100

    # make sure that the pred_label is 1 or 0
    if pred_label != 0 and pred_label != 1:
        return -100

    # invert confidence is pred_label is incorrect
    if pred_label != true_label:
        confidence = - confidence

    # true state is healthy
    if true_label == 0:
        score = confidence

    # true state is faulty
    else:
        if confidence >= 0:
            score = confidence
        else:
            score = 4 * confidence ** 11 + 1.0 * confidence
    return score

In [135]:
scores_cls = []
scores_rgr = []
for item in sample(train_y_hat, 5000):
    pdf_args = item[1]['pdf_args']
    pdf_args['scale'] = pdf_args['scale'] if pdf_args['scale'] > 0.3989 else 0.3989  # Trick
    true_target = train_y.iloc[int(item[0])]['trq_margin']
    true_label = int(train_y.iloc[int(item[0])]['faulty'])
    scores_rgr.append(score_rgr(item[1]['pdf_type'], pdf_args, true_target))
    scores_cls.append(score_cls(item[1]['class'], item[1]['class_conf'], true_label))
sum(scores_rgr) / len(scores_rgr), sum(scores_cls) / len(scores_cls)

(0.8459417289982033, 0.9436178707180061)

The error was on regression: I didn't de-normalized the predicted std, but only the mean

In [138]:
for file in ['validation_submission', 'submission']:
    with open(f'{file}.json') as fp:
        train_y_hat: dict = json.load(fp)
    for key in train_y_hat.keys():
        train_y_hat[key]['pdf_args']['scale'] = train_y_hat[key]['pdf_args']['scale'] if train_y_hat[key]['pdf_args'][
                                                                                             'scale'] > 0.3989 else 0.3989
    with open(f'{file}_trick.json', 'w') as fp:
        json.dump(train_y_hat, fp)